# Previsão de Inadimplência de Crédito — Regressão Logística

**Objetivo:** Construir um classificador binário para prever se um cliente vai entrar em default no crédito concedido.

**Dataset:** Transações hipotéticas de um banco fictício, via Kaggle.  
Colocar os arquivos em `../data/raw/` antes de executar (ver README).

**Fluxo do notebook:**
1. Carregamento dos dados
2. Análise exploratória (EDA)
3. Pré-processamento
4. Divisão treino/teste e escalamento
5. Treinamento do modelo
6. Avaliação completa
7. Comparação com baseline do banco

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
)

## 1. Carregamento dos Dados

Os arquivos `Training_dataset_Original.csv` e `Data_Dictionary.csv` devem estar em `../data/raw/`.  
Os nomes das colunas são lidos do dicionário de dados e renomeados para nomes curtos e descritivos.

In [ ]:
COLUMN_RENAME = {
    'Application ID (primary key)': 'id',
    'Indicator for default': 'default',
    "Credit worthiness score calculated on the basis of borrower's credit history": 'historic_credit_score',
    'Sum of amount due on active credit cards (in $)': 'total_credit_cards_amount',
    'Annual income (in $)': 'annual_income',
    'Estimated market value of a properety owned/used by the borrower (in $)': 'collateral_mkt_value',
    'Maximum of credit available on all active credit lines (in $)': 'credit_limit',
    'Number of active credit cards on which full credit limit is utilized by the borrower': 'number_cards_w_limit_fully_used',
    'Average utilization of line on all active credit cards activated in last 1 year (%)': 'avg_card_utilization_last_1y',
}

FEATURES = [
    'historic_credit_score',
    'total_credit_cards_amount',
    'annual_income',
    'collateral_mkt_value',
    'credit_limit',
    'number_cards_w_limit_fully_used',
    'avg_card_utilization_last_1y',
]
TARGET = 'default'


def load_data(raw_dir: str = '../data/raw') -> pd.DataFrame:
    """Carrega o CSV de treino e renomeia colunas via dicionário de dados."""
    df = pd.read_csv(f'{raw_dir}/Training_dataset_Original.csv')
    dicionario = pd.read_csv(f'{raw_dir}/Data_Dictionary.csv')
    df = df.drop('index', axis=1)
    df.columns = dicionario['Definition']
    df = df[list(COLUMN_RENAME.keys())].rename(columns=COLUMN_RENAME)
    print(f'Dados carregados: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
    return df


df = load_data()
df.head()

## 2. Análise Exploratória (EDA)

Antes de qualquer decisão de pré-processamento, examinamos:
- Distribuição da variável alvo (desbalanceamento de classes)
- Taxa de valores ausentes por feature
- Relação entre missingness e taxa de default (para decidir estratégia de imputação)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax = sns.countplot(x=df[TARGET], palette='Set2')
ax.set_title('Distribuição de Resultado das Concessões de Crédito', fontsize=14)
ax.set_xlabel('Pagamento (0) ou Default (1)', labelpad=10)
ax.set_ylabel('Ocorrências', labelpad=10)
for p in ax.patches:
    ax.annotate(
        f'{100 * p.get_height() / len(df):.1f}%',
        (p.get_x() + p.get_width() / 2, p.get_height() + 400),
        ha='center',
    )
plt.tight_layout()
plt.show()

default_rate = df[TARGET].mean()
print(f'Taxa de default: {default_rate:.1%}  |  Classe majoritária: {1 - default_rate:.1%}')
print('Atenção: dataset desbalanceado — acurácia sozinha é métrica enganosa.')

In [ ]:
def analyze_missingness(df: pd.DataFrame, features: list, target: str) -> pd.DataFrame:
    """
    Analisa valores ausentes e sua relação com a variável alvo.
    Ajuda a decidir estratégia de imputação: se a taxa de default difere
    entre registros com e sem dados, a ausência é um sinal preditivo.
    """
    # Normalizar strings de missingness para NaN antes de contar
    df_check = df[features].replace({'missing': np.nan, 'na': np.nan})
    df_check = df_check.apply(pd.to_numeric, errors='coerce')

    rows = []
    for col in features:
        mask_missing = df_check[col].isna()
        n_miss = mask_missing.sum()
        pct_miss = n_miss / len(df) * 100
        dr_miss = df.loc[mask_missing, target].mean() if n_miss > 0 else None
        dr_present = df.loc[~mask_missing, target].mean()
        rows.append({
            'feature': col,
            'n_missing': int(n_miss),
            'pct_missing': round(pct_miss, 1),
            'default_rate_missing': round(dr_miss, 3) if dr_miss is not None else '-',
            'default_rate_present': round(dr_present, 3),
        })
    return pd.DataFrame(rows).set_index('feature')


missing_stats = analyze_missingness(df, FEATURES, TARGET)
print('Análise de valores ausentes:')
missing_stats

## 3. Pré-processamento

**Decisões de imputação:**
- `collateral_mkt_value` — muitos 'missing'; como é valor de imóvel, ausência indica que o cliente não possui imóvel. Substituímos por 0 e adicionamos indicador binário `has_collateral`.
- `total_credit_cards_amount`, `credit_limit` — strings 'missing'/'na' são tratadas como 0 (ausência de cartões ativos).
- `number_cards_w_limit_fully_used` — strings 'na' tratadas como 0.
- `avg_card_utilization_last_1y` — NaN imputados com mediana (não 0, pois 0 é um valor real e válido).

**Tipos:**
- `avg_card_utilization_last_1y` permanece `float64` — truncar para int descartaria as casas decimais.

In [ ]:
def preprocess(df: pd.DataFrame) -> tuple:
    """
    Seleciona features, trata strings de missingness, imputa valores ausentes
    e retorna X (array float64) e y (array int).

    Retorna
    -------
    X : np.ndarray, shape (n_samples, n_features)
    y : np.ndarray, shape (n_samples,)
    feature_names : list[str]
    """
    q = df.copy()

    # -- Tratar strings de missingness como NaN --------------------------------
    str_cols = ['total_credit_cards_amount', 'credit_limit',
                'number_cards_w_limit_fully_used', 'collateral_mkt_value',
                'historic_credit_score']
    for col in str_cols:
        q[col] = q[col].replace({'missing': np.nan, 'na': np.nan})
        q[col] = pd.to_numeric(q[col], errors='coerce')

    # -- collateral: ausência = sem imóvel (0) + indicador binário -------------
    q['has_collateral'] = q['collateral_mkt_value'].notna().astype(int)
    q['collateral_mkt_value'] = q['collateral_mkt_value'].fillna(0)

    # -- Imputar restantes com 0 (ausência de cartão/limite = 0 faz sentido) --
    for col in ['total_credit_cards_amount', 'credit_limit',
                'number_cards_w_limit_fully_used', 'historic_credit_score']:
        q[col] = q[col].fillna(0)

    # -- avg_card_utilization: imputar com mediana (0 é valor real e válido) --
    median_util = q['avg_card_utilization_last_1y'].median()
    q['avg_card_utilization_last_1y'] = q['avg_card_utilization_last_1y'].fillna(median_util)

    # -- Tipos (preserva float onde necessário) --------------------------------
    int_cols = ['historic_credit_score', 'total_credit_cards_amount', 'annual_income',
                'collateral_mkt_value', 'credit_limit', 'number_cards_w_limit_fully_used',
                'has_collateral']
    for col in int_cols:
        q[col] = q[col].astype('int64')
    # avg_card_utilization_last_1y permanece float64

    feature_names = FEATURES + ['has_collateral']
    X = q[feature_names].astype('float64').values
    y = q[TARGET].values

    print(f'Features: {feature_names}')
    print(f'X shape: {X.shape}  |  Taxa de default: {y.mean():.1%}')
    return X, y, feature_names


X, y, feature_names = preprocess(df)

## 4. Divisão Treino/Teste e Escalamento

**`stratify=y`** preserva a proporção de defaults em ambos os splits — essencial para datasets desbalanceados.

**Escalamento correto:**  
O `StandardScaler` é fitado *apenas* no conjunto de treino (`fit_transform`).  
No conjunto de teste, aplicamos apenas `.transform()` — sem refitar — para que as estatísticas
de normalização (média, desvio padrão) sejam as do treino, não do teste.  
Fitar um scaler separado no teste constitui **data leakage**: o modelo indiretamente "vê"
a distribuição do teste durante o pré-processamento.

In [ ]:
def split_and_scale(X, y, test_size: float = 0.2, random_state: int = 42):
    """
    Split estratificado + StandardScaler fitado apenas no treino.

    Retorna
    -------
    X_train_s, X_test_s : arrays escalados
    y_train, y_test     : arrays de labels
    scaler              : StandardScaler fitado (salvar para inferência em produção)
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)  # aprende média/std do treino
    X_test_s = scaler.transform(X_test)         # aplica estatísticas do treino ao teste

    print(f'Treino: {X_train_s.shape[0]:,} amostras  |  Teste: {X_test_s.shape[0]:,} amostras')
    print(f'Default no treino: {y_train.mean():.1%}  |  Default no teste: {y_test.mean():.1%}')
    return X_train_s, X_test_s, y_train, y_test, scaler


X_train, X_test, y_train, y_test, scaler = split_and_scale(X, y)

## 5. Treinamento do Modelo

Regressão logística com regularização L2 (padrão do sklearn).  
Próximos passos: cross-validation e tuning de `C` (força de regularização).

In [ ]:
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)
print('Modelo treinado.')

## 6. Avaliação Completa

Com ~75% de classe negativa, um modelo que *nunca* prevê default teria ~75% de acurácia.  
Por isso reportamos precision, recall, F1-score e AUC-ROC além da acurácia.

- **Precision** — dentre os previstos como default, quantos realmente são?
- **Recall** — dentre os que realmente são default, quantos o modelo captura?
- **AUC-ROC** — capacidade geral de separação das classes (1.0 = perfeito, 0.5 = aleatório)
- **Curva PR** — mais informativa que ROC para datasets desbalanceados

In [ ]:
def evaluate_classifier(model, X_test, y_test, feature_names=None):
    """Avaliação completa: métricas textuais + 3 gráficos."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # -- Métricas textuais ---------------------------------------------------
    print('=' * 55)
    print('  Relatório de Classificação')
    print('=' * 55)
    print(classification_report(y_test, y_pred,
                                 target_names=['Sem Default', 'Default']))

    auc_roc = roc_auc_score(y_test, y_prob)
    avg_prec = average_precision_score(y_test, y_prob)
    acuracia = accuracy_score(y_test, y_pred)
    print(f'Acurácia:          {acuracia:.4f}')
    print(f'AUC-ROC:           {auc_roc:.4f}')
    print(f'Average Precision: {avg_prec:.4f}')
    print(f'  (baseline AP para {y_test.mean():.1%} de positivos = {y_test.mean():.4f})')

    # -- Gráficos ------------------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Avaliação do Modelo — Regressão Logística', fontsize=13)

    # 1. Matriz de confusão
    cm = confusion_matrix(y_test, y_pred)
    cm_pct = cm / cm.sum() * 100
    disp = ConfusionMatrixDisplay(cm, display_labels=['Sem Default', 'Default'])
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[0].text(j, i + 0.35, f'({cm_pct[i, j]:.1f}%)',
                         ha='center', va='center', fontsize=9, color='black')
    axes[0].set_title('Matriz de Confusão')

    # 2. Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {auc_roc:.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Baseline aleatório')
    axes[1].set_xlabel('Taxa de Falsos Positivos')
    axes[1].set_ylabel('Taxa de Verdadeiros Positivos')
    axes[1].set_title('Curva ROC')
    axes[1].legend(loc='lower right')
    axes[1].grid(alpha=0.3)

    # 3. Curva Precision-Recall
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    axes[2].plot(recall, precision, lw=2, label=f'AP = {avg_prec:.4f}')
    axes[2].axhline(y=y_test.mean(), color='k', linestyle='--', lw=1,
                    label=f'Baseline = {y_test.mean():.4f}')
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].set_title('Curva Precision-Recall')
    axes[2].legend(loc='upper right')
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('../reports/figures/evaluation.png', dpi=150, bbox_inches='tight')
    plt.show()

    # -- Importância das features (coeficientes) ----------------------------
    if feature_names:
        coef_df = pd.DataFrame({
            'feature': feature_names,
            'coeficiente': model.coef_[0],
        }).sort_values('coeficiente', key=abs, ascending=False)
        print('\nImportância das features (coeficientes do modelo):')
        print(coef_df.to_string(index=False))

    return {'accuracy': acuracia, 'auc_roc': auc_roc, 'avg_precision': avg_prec}


metricas = evaluate_classifier(modelo, X_test, y_test, feature_names)

## 7. Comparação com Baseline do Banco

O baseline é a taxa de acerto que a área de crédito teria se *nunca aprovasse nenhum default*  
(i.e., a proporção da classe majoritária no dataset).

In [ ]:
def comparar_com_baseline(df: pd.DataFrame, metricas: dict, target: str = 'default'):
    """Compara a acurácia do modelo com o baseline da área de crédito."""
    eficacia_banco = 1 - df[target].mean()
    eficacia_modelo = metricas['accuracy']
    alfa = eficacia_modelo / eficacia_banco - 1

    sinal = 'mais' if alfa > 0 else 'menos'
    print(f'Eficiência do modelo:       {eficacia_modelo:.4%}')
    print(f'Eficiência da área crédito: {eficacia_banco:.4%}')
    print(f'\nO modelo foi {abs(alfa):.4%} {sinal} eficiente que o baseline.')
    print('\nNota: AUC-ROC e Average Precision são métricas mais relevantes para'
          ' avaliar modelos de crédito desbalanceados.')


comparar_com_baseline(df, metricas)